# Lab 3 — Random Kicks: One Path Is Unpredictable, Its Distribution Is Not

**ROB7104 / ROB8104 Mathematics for Robotics · stochastic differential equations, following Higham (2001)**

This notebook is a reading. There is no code to run and nothing to hand in. Every idea comes with a short video or figure, and most have a live version on the lab's web page, where you can turn the camera and move the sliders yourself:

**Interactive page:** [realgaule.github.io/rob7104/lab3](https://realgaule.github.io/rob7104/lab3/)

It follows D. J. Higham, *An algorithmic introduction to numerical simulation of stochastic differential equations*, SIAM Review 43 (2001), and Lectures 6 and 10 (random walks, white noise, SDEs, Fokker–Planck equations).

> If a video does not play inside your notebook viewer, open `Lab3.html` (same folder) in a browser.

## 0. Particles and densities

<video controls loop muted playsinline width="100%" src="media/s00_hero.mp4" poster="media/s00_hero_poster.png"></video>

Six hundred small robots wander over a table. Each one moves at random: nobody could say where a particular robot will be in ten seconds. Then two lamps switch on. Every robot follows the same simple rule, "drift a little toward brighter light, keep wandering", and within half a minute the swarm has split into two crowds under the lamps.

That is the theme of this lab:

> **One path is unpredictable. The distribution of many paths is predictable.**

There are two ways to describe the same swarm.

- **Particles.** Each robot follows a **stochastic differential equation** (SDE)
  $$dX = f(X)\,dt + g(X)\,dW,$$
  where $f$ is the deliberate drift (toward the light), $g$ the size of the random kicks, and $W$ a Wiener process (Brownian motion).
- **Density.** The density $p(x,t)$ of the whole swarm obeys a deterministic partial differential equation, the **Fokker–Planck equation** (for the Itô reading of the SDE, Lecture 6):
  $$\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x}\big(f\,p\big) + \frac12\,\frac{\partial^2}{\partial x^2}\big(g^2\,p\big).$$

Swarm roboticists use exactly this pair: they program each robot with a local random rule, and predict and design the behaviour of the swarm with the Fokker–Planck equation. In every video below, the robots are the particles, and a translucent **density wall** beside them shows their histogram (blue) together with the density predicted by theory (yellow or cyan).

**Where things happen.** Parts 1–7 take place in a long **corridor**, so the position $x$ is one number and the density is a curve. Parts 8–9 open up to the whole table, a plane.

---

# Part 1 — Brownian motion

## 1.1 How long should a random step be?

A robot that wanders at random takes a small random step every $\Delta t$ seconds. If the steps are independent with mean zero, what size should they have, so that the motion does not depend on how finely we chop time?

Take $N$ steps of size $c\,\Delta t^{\,p}\,Z_j$ on $[0,1]$, with $Z_j \sim N(0,1)$ and $\Delta t = 1/N$. The position at time 1 is a sum of $N$ independent terms, so

$$
\mathrm{Var}\,X(1) = N\,c^2\,\Delta t^{\,2p} = c^2\,\Delta t^{\,2p-1}.
$$

Only $p = \tfrac12$ gives an answer that does not depend on $\Delta t$.

<video controls loop muted playsinline width="100%" src="media/s01_sqrtdt.mp4" poster="media/s01_sqrtdt_poster.png"></video>

*Three corridors of 200 robots, as the number of steps doubles from 2 to 256. Top: steps $\propto \Delta t$; the spread dies (variance 0.024 to 0.0002 m²). Middle: steps $\propto \sqrt{\Delta t}$; the spread stays at 0.048 m². Bottom: steps $\propto \Delta t^{1/4}$; the spread explodes (0.07 to 0.77 m²).*

So random kicks must scale like $\sqrt{\Delta t}$. In the limit this defines the **Wiener process** $W(t)$: $W(0) = 0$, independent increments, and $W(t+\Delta t) - W(t) \sim N(0, \Delta t)$, so a typical increment has size $\sqrt{\Delta t}$ (Higham §2). The paths are continuous but so rough that they have no derivative anywhere.

> **Takeaway.** Random kicks must scale like $\sqrt{\Delta t}$; any other choice makes the motion vanish or blow up as the step shrinks.
> ▶ Try it: [How long should a step be?](https://realgaule.github.io/rob7104/lab3/#d11)

## 1.2 A slice through all paths is a density

Release 400 robots at the centre of a dark corridor. Each one diffuses: $dX = \sqrt{2D}\,dW$ with $D = 0.01$ m²/s. One robot's path is unpredictable, but at any fixed time $t$ the positions of all robots form a histogram, and that histogram is the density of $X(t)$, here $N(0, 2Dt)$. It spreads exactly as the heat equation says.

<video controls loop muted playsinline width="100%" src="media/s01_slice.mp4" poster="media/s01_slice_poster.png"></video>

*Left inset: 40 of the paths, with the current time as a pink slice. On the wall: the histogram of all 400 robots at that slice against $N(0, 2Dt)$. Right inset: how far a histogram is from the true density, against the number of robots.*

After 12 s the robots' variance is 0.235 m² against the exact $2Dt = 0.240$ m². The histogram's error shrinks like $1/\sqrt M$ with the number $M$ of robots (fitted slope $-0.50$): to halve the error you need four times as many robots. This is the Monte Carlo rate, and it will return in Part 5.

> **Takeaway.** Cut through all the paths at one time and you get the density; its accuracy improves only like $1/\sqrt{M}$.
> ▶ Try it: [A slice is a density](https://realgaule.github.io/rob7104/lab3/#d12)

---

# Part 2 — Itô or Stratonovich: where you evaluate matters

## 2.1 Two facts behind Itô calculus

Ordinary calculus drops squares of small increments: $(\Delta t)^2$ is negligible next to $\Delta t$. For Brownian motion that fails. Each $(\Delta W)^2$ has mean $\Delta t$, so over $[0,T]$

$$
\sum_j (\Delta W_j)^2 \;\to\; T \qquad (\text{with spread } \sqrt{2/N}\,T),
$$

on every path, not just on average. This is what the shorthand $(dW)^2 = dt$ means.

A consequence: stochastic integrals depend on *where in each step* the integrand is evaluated (Higham §3). For $\int_0^T W\,dW$, evaluating $W$ at the fraction $\theta$ of each step gives a sum whose mean is $\theta\,T$. The **Itô** integral uses the left end ($\theta = 0$): $\int_0^T W\,dW = \tfrac12 W(T)^2 - \tfrac12 T$. The **Stratonovich** integral uses the midpoint ($\theta = \tfrac12$): $\tfrac12 W(T)^2$, as in ordinary calculus.

<img src="media/s02_ito_sums.png" width="100%">

*Left: $\sum(\Delta W)^2$ settles at $T = 1$ on every path, and its spread shrinks like $\sqrt{2/N}$. Right: the mean of the $\theta$-evaluated sum over 10 000 paths lies on $\theta T$ (Itô: 0.010; Stratonovich: 0.510).*

So an SDE $dX = g(X)\,dW$ whose noise amplitude $g$ depends on $X$ is not yet a complete model: you must also say where $g$ is evaluated. Is this just a convention? For robots, no.

## 2.2 When the noise depends on the state

A classic swarm behaviour is **photokinesis**: robots do not steer toward the light at all, they just *slow down* in it. Each robot drives at speed $v(x) = v_0\,(1 - s\,I(x))$, where $I(x)$ is the light level, and reverses direction at random, at a constant rate $a$. Over many reversals this is diffusion with a diffusivity that depends on the position:

$$
dX = \sqrt{2D(X)}\;dW, \qquad D(x) = \frac{v(x)^2}{2a}.
$$

The noise amplitude depends on $X$, so the interpretation matters, and it decides where the swarm gathers. With no drift and walls at both ends, the stationary density is

$$
\text{Itô: } p \propto \frac{1}{D} \propto \frac{1}{v^2}, \qquad\qquad \text{Stratonovich: } p \propto \frac{1}{\sqrt{D}} \propto \frac{1}{v}.
$$

Both say the robots crowd where they are slow, but by different amounts. Which one do real robots follow?

<video controls loop muted playsinline width="100%" src="media/s02_interpretation.mp4" poster="media/s02_interpretation_poster.png"></video>

*Three corridors with the same lamp: (1) the robots themselves, driving and reversing as described; (2) the SDE read as Itô and simulated with Euler–Maruyama; (3) the same SDE read as Stratonovich and simulated with Heun's method. Each wall shows both predictions: $1/v$ (yellow) and $1/D$ (pink dashed). $v_0 = 1$ m/s, $s = 0.6$, $a = 4$ /s. Played ten times as fast.*

In a long, fully settled run with 20 000 robots, the real robots' density is within 1.3% of the Stratonovich prediction $1/v$ and 42% away from the Itô prediction. The crowd under the lamp is 2.5 times as dense as at the ends; Stratonovich predicts 2.47, Itô 6.1.

**Why Stratonovich.** In the steady state the right-movers and left-movers have equal density and carry no net flow, so $\frac{d}{dx}\big(v\,p\big) = 0$ and $p \propto 1/v$ exactly. The Itô reading treats each kick as if its size were fixed at the start of the step; but a real robot changes speed *during* its run as it enters the light. The Stratonovich (midpoint) reading captures that, and the extra robots that Itô piles into the light do not exist.

> **Takeaway.** When the noise depends on the state, Itô and Stratonovich are different models, and they put the swarm in different places. The physics of the robot decides which one is right.
> ▶ Try it: [Where do slow robots gather?](https://realgaule.github.io/rob7104/lab3/#d21)

---

# Part 3 — Euler–Maruyama

An SDE has to be simulated step by step. The **Euler–Maruyama** method (Higham §4) freezes $f$ and $g$ at the start of each step:

$$
X_{j} = X_{j-1} + f(X_{j-1})\,\Delta t + g(X_{j-1})\,\Delta W_j,\qquad \Delta W_j = \sqrt{\Delta t}\,Z_j,\ Z_j \sim N(0,1).
$$

Evaluating $g$ at the start of the step makes it an Itô method. (For a Stratonovich SDE, either convert it first by adding the drift $\tfrac12 g\,g'$, or use a midpoint method such as Heun's, as in Part 2.)

**Higham's test equation.** For $dX = \lambda X\,dt + \mu X\,dW$ the exact solution is $X(t) = X_0\,e^{(\lambda - \mu^2/2)t + \mu W(t)}$, a lognormal random variable.

<img src="media/s03_em_linear.png" width="100%">

*Left: one Brownian path, the exact solution and Euler–Maruyama with three step sizes ($\lambda = 2$, $\mu = 1$). Right: 20 000 Euler–Maruyama paths at $t = 1$ against the exact lognormal density.*

Two things to notice. On a single path, the error does not always shrink with the step (0.60, 0.62, 0.14 here); only on average does it behave, which is Part 5's subject. And the mean $\mathbb E\,X(1) = e^{\lambda} = 7.39$ lies far above the median $e^{\lambda - \mu^2/2} = 4.48$: the mean is pulled up by rare large paths, while a typical path grows only at the rate $\lambda - \mu^2/2$. Part 8 explains the $-\mu^2/2$.

**On the robots.** Robots that steer toward a lamp ("phototaxis") drift up the light gradient: $dX = \kappa\,I'(X)\,dt + \sigma\,dW$. Euler–Maruyama moves each robot by its drift and a kick, and the histogram of the swarm follows the Fokker–Planck solution.

<video controls loop muted playsinline width="100%" src="media/s03_em.mp4" poster="media/s03_em_poster.png"></video>

*Robots start at the left and steer to a lamp at $x = 0.5$ m ($\kappa = 0.08$ m²/s, $\sigma = 0.12$ m/$\sqrt{\text{s}}$, $\Delta t = 0.05$ s, 3000 robots). On the wall: their histogram against the Fokker–Planck solution (cyan).*

About 15% of the robots never find the lamp: far from it the light is almost flat, the drift almost zero, and they wait at the far wall. The Fokker–Planck solution predicts the same small bump. A model that steers by the light gradient alone is short-sighted, and both views agree on that.

> **Takeaway.** Euler–Maruyama moves every particle by $f\,\Delta t + g\sqrt{\Delta t}\,Z$; run it for many robots and the histogram tracks the density.
> ▶ Try it: [Euler–Maruyama and the density](https://realgaule.github.io/rob7104/lab3/#d31)

---

# Part 4 — Fokker–Planck: watching the density directly

The particle view needs thousands of simulated robots and still gives a noisy histogram. The density view gives the answer directly: the Fokker–Planck equation (Lecture 6) moves probability by the drift and spreads it by the noise,

$$
\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x}\big(f\,p\big) + \frac12\frac{\partial^2}{\partial x^2}\big(g^2 p\big).
$$

When the drift comes from a landscape and the noise is constant, the density settles to a **stationary** shape. For robots steering up the light, $f = \kappa\,I'(x)$ and $g = \sigma$, setting the flux to zero gives

$$
p_\infty(x) \;\propto\; \exp\!\Big(\frac{2\kappa\,I(x)}{\sigma^2}\Big):
$$

more robots where it is brighter, sharpened by a strong pull $\kappa$ and blurred by strong noise $\sigma$. With two lamps the light has two peaks, the "landscape" is a **double well**, and the density must split.

<video controls loop muted playsinline width="100%" src="media/s04_fokker_planck.mp4" poster="media/s04_fokker_planck_poster.png"></video>

*4000 robots start in the dark middle of the corridor between two lamps ($\kappa = 0.08$ m²/s, $\sigma = 0.2$ m/$\sqrt{\text{s}}$). On the wall: the robots' histogram (blue), the Fokker–Planck solution (cyan) and the stationary density (pink dashed). First the lamps are equal; then the right one is turned up from 0.8 to 1.1. The playback speeds up to 50 times.*

| | Left half | Right half |
|---|---|---|
| Equal lamps: robots / Fokker–Planck / stationary | 51% / 50% / 50% | 49% / 50% / 50% |
| Right lamp brighter: robots / Fokker–Planck / stationary | 27% / 28% / 27% | 73% / 72% / 73% |

The robots and the Fokker–Planck solution agree at every instant, not only at the end: their share in the right half never differs by more than 1.4 percentage points. Moving robots from the dim lamp to the bright one is slow: a robot must cross the dark middle against the pull of its own lamp, and on average that takes about two minutes here. The density equation shows this slow traffic as a slowly shifting balance between the two bumps.

**Higham's population model.** The same machinery works for any SDE. Higham's logistic population model $dX = rX(K - X)\,dt + \beta X\,dW$ settles to a Gamma density:

<img src="media/s04_population.png" width="100%">

*$r = 2$, $K = 1$, $\beta = 0.25$, at $t = 3$: 20 000 Euler–Maruyama paths, the Fokker–Planck solution, and the stationary Gamma density (mean 0.98, standard deviation 0.12 in all three).*

> **Takeaway.** The density obeys a deterministic equation of its own; one bump splits into two and settles at $p_\infty \propto e^{2\kappa I/\sigma^2}$, in step with the robots at every moment.
> ▶ Try it: [Two lamps, one swarm](https://realgaule.github.io/rob7104/lab3/#d41)

---

# Part 5 — Can we trust the simulation? Strong and weak convergence

Every picture so far came from a simulation with a finite step $\Delta t$. How wrong is it, and does the answer depend on what we want to know?

## 5.1 Two questions about the same swarm

A swarm engineer can ask two different things.

- **Where will *this* robot be?** Then each simulated path must stay close to the true path driven by the same noise. The error is measured path by path, $\mathbb E\,|X_L - X(T)|$. This is **strong** convergence.
- **How will the swarm be spread?** Then only the distribution must be right, for example its mean, $|\mathbb E\,X_L - \mathbb E\,X(T)|$. This is **weak** convergence.

A method has strong order $\gamma$ if the strong error shrinks like $\Delta t^{\gamma}$, and weak order $\beta$ if the weak error shrinks like $\Delta t^{\beta}$ (Higham §5). For Euler–Maruyama, $\gamma = \tfrac12$ and $\beta = 1$.

<video controls loop muted playsinline width="100%" src="media/s05_strong_weak.mp4" poster="media/s05_strong_weak_poster.png"></video>

*The same swarm of robots that slow down in light, simulated twice with the same random kicks: once with a coarse step $\Delta t = 2$ s and once with a fine reference step of $1/64$ s. Front corridor: eight robots (pink) next to their reference selves (yellow ghosts). Back corridor: the density wall, coarse (blue) against reference (yellow). Played four times as fast.*

After 40 s the coarse robots are on average 8.5 cm away from their reference selves, yet the share of robots under the lamp is 35.9% against 35.4%. The coarse simulation is poor at tracking any one robot and good at predicting the swarm.

### The orders, measured

<img src="media/s05_orders.png" width="100%">

On Higham's test equation $dX = 2X\,dt + X\,dW$ the fitted orders are 0.52 (strong) and 1.01 (weak), as theory says. The third panel shows something surprising: replacing every Gaussian kick by a coin flip $\pm\sqrt{\Delta t}$ makes each path wrong in a new way, but the mean is exactly the same as with Gaussian kicks (6.58 at $\Delta t = 1/16$, against the true 7.39 that both approach as $\Delta t \to 0$). Weak accuracy only cares about the distribution of the kicks to the right order.

**Measuring a weak error is itself a sampling problem.** A Monte Carlo mean from $M$ paths has a random error of order $1/\sqrt M$. If that is larger than the weak error you are trying to see, you see noise.

<img src="media/s05_sampling.png" width="100%">

With 1 000 paths the weak error is hidden by sampling noise at three of seven step sizes; with 100 000 paths it shows at all seven.

> **Takeaway.** Halving $\Delta t$ improves each path by about $\sqrt 2$ but halves the error of the mean. For a swarm, the weak order is usually the one that matters.
> ▶ Try it: [One swarm, two questions](https://realgaule.github.io/rob7104/lab3/#d51)

# Part 6 — Milstein's method: one extra term

Euler–Maruyama freezes the noise amplitude $g(X)$ at the start of each step. When $g$ depends on the state, that is the dominant error. Expanding $g$ to first order inside the step and using $(\Delta W)^2 \approx \Delta t$ gives one correction term (Higham §6):

$$
X_{j} = X_{j-1} + f\,\Delta t + g\,\Delta W_j + \tfrac12\,g\,g'\,\big((\Delta W_j)^2 - \Delta t\big).
$$

It raises the strong order from $\tfrac12$ to $1$; the weak order stays 1.

<img src="media/s06_milstein.png" width="100%">

*Left: Higham's test equation, Euler–Maruyama order 0.53 against Milstein 0.97. Middle: the error of each single path at $\Delta t = 2^{-6}$ (median 0.24 for Euler–Maruyama, 0.056 for Milstein). Right: the robots that slow down in light.*

For the robots the noise amplitude is $g = v(x)/\sqrt a$: it changes with the light, so $g' \ne 0$ and the extra term matters. Milstein reaches order 1.00 on the swarm in an open corridor. With reflecting walls it drops to 0.60: bouncing off a wall is not smooth, and the expansion behind the extra term assumes smoothness. If the noise amplitude were constant, $g' = 0$ and Milstein would be Euler–Maruyama.

> **Takeaway.** One extra term, $\tfrac12 g g'((\Delta W)^2 - \Delta t)$, makes each path accurate to order 1, as long as the model is smooth.

# Part 7 — Stability: will it blow up?

## 7.1 Almost every path dies, yet the mean square explodes

Higham's §7 example is $dX = \lambda X\,dt + \mu X\,dW$ with $\lambda = \tfrac12$ and $\mu = \sqrt 6$. Its exact solution is $X(t) = e^{(\lambda - \mu^2/2)t + \mu W(t)}$, so $\ln X$ drifts down at the rate $\lambda - \mu^2/2 = -2.5$: almost every path goes to zero. Yet $\mathbb E\,X^2 = e^{(2\lambda + \mu^2)t} = e^{7t}$ grows without bound.

<video controls loop muted playsinline width="100%" src="media/s07_lottery.mp4" poster="media/s07_lottery_poster.png"></video>

Both are true at once because $\mathbb E\,X^2$ is carried by paths so rare that you never see them. At $t = 3$, 96% of the paths are below 1, while the mass of $x^2p(x)$ sits near $\ln x = 28.5$, far beyond the largest of $10^5$ samples ($\ln x = 12.1$). A Monte Carlo estimate of $\mathbb E\,X^2$ from $10^5$ paths is then useless: it can be too low by a factor of thousands, or too high when one rare path happens to appear. There are two notions of stability: **mean-square** ($\mathbb E\,X^2 \to 0$) and **asymptotic** (almost every path $\to 0$), and they can disagree.

## 7.2 The step size can destroy stability

A stable SDE can become unstable in simulation. For the test equation, one Euler–Maruyama step multiplies $X$ by $1 + \lambda\Delta t + \mu\,\Delta W$, so stability depends only on $x = \Delta t\,\lambda$ and $y = \Delta t\,\mu^2$:

<img src="media/s07_regions.png" width="100%">

Mean-square stability of the method needs $(1+x)^2 + y < 1$, the blue region on the left. For both of Higham's examples the method is stable at $\Delta t = 1/4$ and not at $\Delta t = 1$; the limits are $\Delta t = 1/3$ and $0.32$.

**On the robots.** Robots steer toward a lamp with a pull proportional to their distance, $f = -kx$ with $k = 0.5$ /s. The real swarm is stable for any $k$. One Euler–Maruyama step multiplies the distance by $1 - k\Delta t$, so the simulation is stable only if $\Delta t < 2/k = 4$ s.

<video controls loop muted playsinline width="100%" src="media/s07_stability.mp4" poster="media/s07_stability_poster.png"></video>

| Step | Fraction of the limit | What the simulation says |
|---|---|---|
| $\Delta t = 0.1$ s | 0.025 | spread 7.8 cm (true 7.7 cm) |
| $\Delta t = 3.6$ s | 0.9 | stable, but spread 24.9 cm: three times too wide |
| $\Delta t = 4.8$ s | 1.2 | the distance grows 1.4 times per step; 94% of the robots are thrown out of the corridor |

The middle row is the subtle one: below the limit the simulation does not blow up, but it is still badly wrong. Stable is not the same as accurate.

> **Takeaway.** A stable system can be simulated into instability by too large a step, and even below the limit a large step distorts the distribution.
> ▶ Try it: [A steep lamp and the step size](https://realgaule.github.io/rob7104/lab3/#d71)

---

# Part 8 — The Itô chain rule

## 8.1 A curved function picks up an extra drift

If $X$ follows $dX = f\,dt + g\,dW$ and we look at a smooth function $V(X)$, ordinary calculus would say $dV = V'\,dX$. But a Taylor expansion to second order keeps a term $\tfrac12 V''\,(dX)^2$, and $(dX)^2 = g^2\,(dW)^2 = g^2\,dt$ is not negligible (Part 2). So

$$
dV = \Big(V' f + \tfrac12\,g^2\,V''\Big)\,dt + V' g\,dW .
$$

The extra drift $\tfrac12 g^2 V''$ is the whole difference between Itô calculus and ordinary calculus (Higham §8, Lecture 6).

**The $-\tfrac12\mu^2$ of Part 3, explained.** For $dX = \lambda X\,dt + \mu X\,dW$ take $V = \ln X$: $V' = 1/X$, $V'' = -1/X^2$, so

$$
d(\ln X) = \big(\lambda - \tfrac12\mu^2\big)\,dt + \mu\,dW .
$$

A typical path grows at the rate $\lambda - \tfrac12\mu^2$, not $\lambda$: the curvature of the logarithm takes away $\tfrac12\mu^2$. That is why the median in Part 3 sat far below the mean.

## 8.2 The distance from the start drifts outward

Release robots together at the centre of the dark table. Each one diffuses in the plane, $dX = \sqrt{2D}\,dW$ with independent kicks in $x$ and $y$. Nothing pushes them in any direction, and indeed the mean position stays at the start. But apply the chain rule to the distance $r = \sqrt{x^2 + y^2}$:

$$
dr = \frac{D}{r}\,dt + \sqrt{2D}\;dW_r ,\qquad \mathbb E\,r^2 = 4Dt .
$$

The distance has a drift $D/r$ although no force acts: random kicks in two directions are more likely to take a robot away from where it stands than straight back. The ordinary chain rule would miss the $D/r$ term and predict that on average every robot stays at distance zero.

<video controls loop muted playsinline width="100%" src="media/s08_chain_rule.mp4" poster="media/s08_chain_rule_poster.png"></video>

*20 000 robots (800 drawn) with $D = 0.004$ m²/s. Rings: the exact mean distance $\mathbb E\,r = \sqrt{\pi D t}$ (yellow) and $\sqrt{\mathbb E\,r^2} = \sqrt{4Dt}$ (dashed). Inset: the distances against the exact Rayleigh density.*

| Time | $\mathbb E\,r$, exact / robots | $\mathbb E\,r^2$, exact $4Dt$ / robots |
|---|---|---|
| 1 s | 0.112 / 0.112 m | 0.016 / 0.016 m² |
| 4 s | 0.224 / 0.225 m | 0.064 / 0.064 m² |
| 16 s | 0.448 / 0.450 m | 0.256 / 0.257 m² |

The mean distance grows like $\sqrt t$, which is why a swarm searching at random covers ground quickly at first and slowly later.

> **Takeaway.** Because $(dW)^2 = dt$, a curved function of the state gains a drift $\tfrac12 g^2 V''$; for random wanderers the distance from the start drifts outward although nothing pushes.
> ▶ Try it: [The distance drifts outward](https://realgaule.github.io/rob7104/lab3/#d81)

---

# Part 9 — Two dimensions: clouds of robots

## 9.1 Holding a spot: a linear SDE keeps a Gaussian

A robot is told to hold a marked spot. Its controller pulls it back like a spring with damping, while random pushes (uneven floor, wheel slip) knock it around:

$$
d\mathbf p = \mathbf v\,dt,\qquad d\mathbf v = (-k\,\mathbf p - c\,\mathbf v)\,dt + G\,d\mathbf W .
$$

This is a linear SDE, $dX = AX\,dt + C\,dW$ with $X = (\mathbf p, \mathbf v)$. For linear SDEs the particle and density views can both be solved exactly (Lecture 6): if the robots start at one point, their positions stay **Gaussian** for all time, with mean $e^{At}X_0$ and a covariance that obeys

$$
\dot\Sigma = A\Sigma + \Sigma A^\top + CC^\top ,
$$

settling at the solution of the **Lyapunov equation** $A\Sigma_\infty + \Sigma_\infty A^\top + CC^\top = 0$.

<video controls loop muted playsinline width="100%" src="media/s09_hold_point.mp4" poster="media/s09_hold_point_poster.png"></video>

*4000 runs (200 drawn) start from the same offset and speed. Yellow: the predicted 1σ and 2σ ellipses at each moment. White: the mean path. Pink dashed: the stationary ellipse from the Lyapunov equation. $k = 1$ /s², $c = 0.5$ /s (damping ratio 0.25), pushes stronger along one direction (30°).*

The mean spirals into the spot while the cloud grows into its final ellipse. Along the push direction the stationary variance is predicted at 0.090 m² and measured at 0.089 m²; across it, 0.0169 against 0.0168. The ellipse is elongated along the direction in which the pushes are strongest; here the controller is the same in every direction, so the ellipse keeps that shape while it grows.

> **Takeaway.** A linear SDE keeps a Gaussian Gaussian: the mean follows the noise-free motion, and the covariance settles at the Lyapunov solution.
> ▶ Try it: [Holding a spot](https://realgaule.github.io/rob7104/lab3/#d91)

## 9.2 Four lamps: Langevin dynamics and annealing

Back to phototaxis, now in the plane, with four lamps of brightness 100%, 60%, 45% and 30%. Each robot moves down the "landscape" $U = -\kappa I$ and is kicked at random:

$$
dX = -\nabla U(X)\,dt + \sqrt{2T}\,dW .
$$

This is **Langevin dynamics**; $T$ plays the role of a temperature. Its Fokker–Planck equation has the stationary density

$$
p_\infty(x) \;\propto\; e^{-U(x)/T},
$$

the **Boltzmann distribution**: every lamp gets robots, but brighter lamps exponentially more.

<video controls loop muted playsinline width="100%" src="media/s09_four_lamps.mp4" poster="media/s09_four_lamps_poster.png"></video>

*4000 robots (600 drawn) all start under the dimmest lamp ($\kappa = 0.02$ m²/s, $T = 0.004$ m²/s). Inset: the share of robots under each lamp (within 0.45 m) against the Boltzmann prediction.*

| | 100% lamp | 60% | 45% | 30% | dark |
|---|---|---|---|---|---|
| Boltzmann $e^{-U/T}$ | 55.6% | 13.1% | 8.0% | 5.1% | 18.2% |
| Robots, averaged over 2000–3000 s | 56.2% | 12.9% | 7.9% | 5.1% | 17.9% |

The noise is what lets robots leave the dim lamp at all: with $T = 0$ they would stay where they started, at the nearest lamp. With noise they explore, and in the long run they sample $e^{-U/T}$, whatever their start.

**Annealing.** Lowering $T$ sharpens $e^{-U/T}$ toward the brightest lamp, but also makes it harder to cross the dark between lamps. Cooling must therefore be slow.

<img src="media/s09_four_lamps_anneal.png" width="100%">

Cooling from $T = 0.008$ to $0.0015$ over 100 s leaves only 5% of the robots under the brightest lamp; most are frozen where they happened to be. Over 3000 s, 75% end there. This is **simulated annealing**: noise plus slow cooling finds the best of several optima, where pure gradient ascent would stop at the nearest one.

**A finite step changes the stationary density.** Euler–Maruyama does not sample $e^{-U/T}$ exactly. For the simplest case, $dX = -X\,dt + \sqrt{2T}\,dW$, the simulated stationary variance is $T/(1 - \Delta t/2)$ instead of $T$: 5% too wide at $\Delta t = 0.1$, 67% at $\Delta t = 0.8$. With the four lamps and a step of 4 s, the brightest lamp holds 40% instead of 56%. Part 7 in another guise: below the stability limit the method is stable but biased.

> **Takeaway.** Noise lets the swarm leave a poor lamp; Langevin dynamics settles in proportion to $e^{-U/T}$, and slow cooling gathers it under the brightest.
> ▶ Try it: [Four lamps](https://realgaule.github.io/rob7104/lab3/#d92)

---

## 10. Summary

$$
\underbrace{dX = f(X)\,dt + g(X)\,dW}_{\text{one robot}} \quad\Longleftrightarrow\quad \underbrace{\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x}\big(f\,p\big) + \frac12\frac{\partial^2}{\partial x^2}\big(g^2 p\big)}_{\text{the whole swarm}}
$$

| Part | The question | The answer | Seen on |
|---|---|---|---|
| 1 Brownian motion | How big is a random kick? | $\sqrt{\Delta t}$; a slice through all paths is a density; accuracy $\propto 1/\sqrt M$ | robots in a dark corridor |
| 2 Itô or Stratonovich | Where is the noise evaluated? | $(dW)^2 = dt$; the reading changes the model; robots that slow down in light follow Stratonovich, $p \propto 1/v$ | three corridors under a lamp |
| 3 Euler–Maruyama | How do we simulate? | $X_j = X_{j-1} + f\,\Delta t + g\,\Delta W_j$ | robots steering to a lamp |
| 4 Fokker–Planck | Where will the swarm be? | a deterministic PDE for the density; $p_\infty \propto e^{2\kappa I/\sigma^2}$ | two lamps |
| 5 Convergence | Is the simulation right? | strong order $\tfrac12$ for each robot, weak order 1 for the swarm | coarse vs fine swarms |
| 6 Milstein | Can each path be order 1? | add $\tfrac12 g g'((\Delta W)^2 - \Delta t)$, if the model is smooth | robots slowed by light |
| 7 Stability | Can a simulation blow up? | yes, if the step is too large; mean-square and pathwise stability differ | robots thrown out of the corridor |
| 8 Itô chain rule | How do functions of $X$ move? | extra drift $\tfrac12 g^2 V''$; the distance drifts outward | robots spreading on the table |
| 9 Two dimensions | What about the plane? | linear SDEs keep Gaussians (Lyapunov); Langevin samples $e^{-U/T}$; anneal slowly | holding a spot; four lamps |

**Notes on the simulations.**

- The robots are drawn in 3D with MuJoCo. Their motion comes from the SDEs in the text, simulated with Euler–Maruyama, Heun or Milstein as stated; the run-and-tumble robots of Part 2 are simulated directly as robots that drive and reverse.
- Robot positions are modelled as diffusing points: the heading of a robot is not tracked. This is the usual swarm-robotics approximation when robots turn often.
- Densities on the walls and maps are histograms of all simulated robots (usually thousands; only some are drawn), compared with exact densities or with Fokker–Planck solutions computed on a fine grid.
- The numbers in the text are computed by the scripts that made the videos.